# 01b — Random-Init Baseline (control for the SALT init)

Builds `neobert_random_init_baseline`: identical to the SALT init in EVERY respect except the
content embeddings are random instead of SALT-projected. This is the control that makes CPT
results interpretable — SALT only "works" if it beats this.

**Held identical to the SALT init (so the only variable is embedding semantics):**
- NeoBERT English encoder (transferred, same weights)
- pruned ViDeBERTa tokenizer (reused from the SALT init artifact, 30,522 vocab)
- special-token embedding rows copied from NeoBERT (structural, language-agnostic)
- decoder = global NeoBERT emb→dec map applied to the embeddings × DECODER_WEIGHT_SCALE
- decoder bias = Vietnamese unigram log-frequency (the prior; both inits get it)
- per-row norm = NeoBERT mean norm (so only DIRECTION differs, not scale)

**The one difference:** content (non-special) rows are random directions, carrying zero
Vietnamese meaning. Step-0 loss therefore ≈ the same unigram floor (~7.3) as the SALT init —
the freq-bias prior dominates step-0 for BOTH — so the experiment is the *training trajectory*,
not step-0. If SALT's semantic embeddings help, its CPT curve / downstream beats this.

`BASELINE_FREQ_BIAS=False` gives the harsher pure-random control (zeroed bias, step-0 ≈ ln V).


In [1]:
%%capture
!pip install -U transformers safetensors huggingface_hub sentencepiece datasets accelerate

In [2]:
import sys, math, json, shutil, glob
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM, AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)
PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
import importlib
import salt3_common as sc; importlib.reload(sc)
import salt3_decoder_variants as sdv; importlib.reload(sdv)
from salt3_common import configure_environment, set_seed, ensure_dir, fit_embedding_to_decoder_map
configure_environment(); set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Config ──────────────────────────────────────────────────────────────────
SOURCE_MODEL_ID    = 'chandar-lab/NeoBERT'
SALT_INIT_NAME     = 'videberta_salt_init_v5_globalmap_freqbias'  # template: tokenizer, patched files, bias
BASELINE_NAME      = 'neobert_random_init_baseline'
DECODER_WEIGHT_SCALE = 0.1            # match the SALT init
BASELINE_FREQ_BIAS   = True           # True = fair control (bias copied verbatim from v5); False = zero bias
# 'neobert_meannorm'  = random directions at NeoBERT's trained mean row norm. FAIR control:
#                       same scale as the SALT init, so the two arms differ ONLY in semantics.
# 'scratch_init_range' = N(0, embedding_init_range) exactly like NeoBERT pretraining from
#                       scratch. NOT scale-matched to SALT (adds a confound) — use only as a
#                       separate from-scratch reference arm.
EMBEDDING_RANDOM_MODE = 'neobert_meannorm'

SALT_INIT_DIR = PROJECT_ROOT / 'init' / SALT_INIT_NAME
SALT_MODEL    = SALT_INIT_DIR / 'model'
INIT_DIR      = ensure_dir(PROJECT_ROOT / 'init' / BASELINE_NAME)
MODEL_DIR     = INIT_DIR / 'model'
print('control of:', SALT_INIT_NAME, '| out:', MODEL_DIR)

Mounted at /content/drive
control of: videberta_salt_init_v5_globalmap_freqbias | out: /content/drive/MyDrive/SALT3/init/neobert_random_init_baseline/model


## 1. Load tensors only — no hub-NeoBERT instantiation

Hub NeoBERT's `model.py` hard-imports xformers, so `AutoModel`/`AutoConfig` with
`trust_remote_code` fails `check_imports` in an xformers-free env (the crash this cell used to
produce). Raw safetensors reads sidestep the dynamic-module machinery entirely; the artifact is
later built from **v5's model dir**, whose body weights are byte-identical to NeoBERT and whose
`model.py`/`rotary.py` already carry the pure-SwiGLU + real-rotary patches.

In [3]:
# NeoBERT embeddings + decoder straight from hub safetensors (no check_imports)
neo_st = load_file(hf_hub_download(SOURCE_MODEL_ID, 'model.safetensors'))
neo_emb = neo_st['model.encoder.weight'].float()
neo_dec = neo_st['decoder.weight'].float()
src_tok = sc.load_tokenizer_no_remote_code(SOURCE_MODEL_ID, INIT_DIR / 'neobert_tokenizer_local')
src_vocab = src_tok.get_vocab()

# v5 artifact: pruned Vietnamese tokenizer + saved tensors (bias copied later)
tgt_tok = AutoTokenizer.from_pretrained(SALT_MODEL, trust_remote_code=True)
tgt_vocab = tgt_tok.get_vocab()
v5_st = load_file(str(SALT_MODEL / 'model.safetensors'))
V, HID = v5_st['model.encoder.weight'].shape
assert V == len(tgt_tok), f'v5 embedding rows {V} != tokenizer {len(tgt_tok)}'

neo_norms = neo_emb.norm(dim=1)
neo_mean_norm = neo_norms[neo_norms > 1e-6].mean().item()
print(f'NeoBERT emb {tuple(neo_emb.shape)} mean-norm {neo_mean_norm:.4f} | target vocab {V} | hidden {HID}')

model.safetensors:   0%|          | 0.00/981M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

NeoBERT emb (30522, 768) mean-norm 1.2064 | target vocab 30522 | hidden 768


## 2. Random content embeddings + copied special rows (norm-matched)

In [4]:
if EMBEDDING_RANDOM_MODE == 'neobert_meannorm':
    # random directions at NeoBERT mean norm -> same scale as the SALT init, zero semantics
    g = torch.Generator().manual_seed(42)
    new_emb = F.normalize(torch.randn(V, HID, generator=g), dim=1) * neo_mean_norm
elif EMBEDDING_RANDOM_MODE == 'scratch_init_range':
    # NeoBERT's own from-scratch recipe: N(0, embedding_init_range)
    init_range = float(json.loads((SALT_MODEL / 'config.json').read_text(encoding='utf-8'))
                       .get('embedding_init_range', 0.02))
    g = torch.Generator().manual_seed(42)
    new_emb = torch.randn(V, HID, generator=g) * init_range
    print(f'scratch init: N(0, {init_range})')
else:
    raise ValueError(EMBEDDING_RANDOM_MODE)

# copy special-token rows from NeoBERT (structural; identical treatment to the SALT init)
special_pairs = []
for key in ('pad_token', 'unk_token', 'cls_token', 'sep_token', 'mask_token', 'bos_token', 'eos_token'):
    tt, st = getattr(tgt_tok, key, None), getattr(src_tok, key, None)
    if tt is None or st is None: continue
    ti, si = tgt_vocab.get(tt), src_vocab.get(st)
    if ti is None or si is None: continue
    new_emb[ti] = neo_emb[si]
    special_pairs.append((key, ti, si))
if tgt_tok.pad_token_id is not None and tgt_tok.pad_token_id < V:
    new_emb[tgt_tok.pad_token_id] = 0.0
print('special rows copied:', [p[0] for p in special_pairs])
print(f'embedding: mean-norm {new_emb.norm(dim=1).mean():.4f} (NeoBERT {neo_mean_norm:.4f})')

special rows copied: ['pad_token', 'unk_token', 'cls_token', 'sep_token', 'mask_token']
embedding: mean-norm 1.2063 (NeoBERT 1.2064)


## 3. Decoder = global emb→dec map (same construction as SALT) + freq bias

In [5]:
emb_to_dec, resid = fit_embedding_to_decoder_map(neo_emb, neo_dec)
print(f'global emb->dec residual: {resid:.3f}')
new_dec = (new_emb.float() @ emb_to_dec) * DECODER_WEIGHT_SCALE
print(f'decoder mean row-norm {new_dec.norm(dim=1).mean():.4f}')

if BASELINE_FREQ_BIAS:
    # copy v5's bias VERBATIM: guarantees the bias is IDENTICAL across arms (recounting
    # from a stream risks drift and wastes the ~20k-doc streaming time)
    new_bias = v5_st['decoder.bias'].float().clone()
    print(f'freq bias copied from v5: range [{new_bias.min():.2f}, {new_bias.max():.2f}]')
else:
    new_bias = torch.zeros(V)
    print('freq bias ZEROED (pure-random control)')

global emb->dec residual: 0.549
decoder mean row-norm 0.0384
freq bias copied from v5: range [-16.78, -3.28]


## 4. Build the artifact from v5's model dir

Copies v5's `model/` (patched `model.py`/`rotary.py`, config, tokenizer files, NeoBERT body
weights — all byte-identical) and swaps the three tensors. No model instantiation, no
`save_pretrained`, no xformers anywhere.

In [6]:
sdv.write_artifact(SALT_INIT_DIR, INIT_DIR, {
    'model.encoder.weight': new_emb,
    'decoder.weight': new_dec,
    'decoder.bias': new_bias,
}, {
    'init_name': BASELINE_NAME, 'type': 'random_baseline_control',
    'source_model': SOURCE_MODEL_ID, 'tokenizer_from': SALT_INIT_NAME,
    'target_vocab_size': int(V),
    'embedding_init': f'random_{EMBEDDING_RANDOM_MODE}',
    'special_token_init': 'copied_from_neobert_specials',
    'decoder_init': 'global_emb_to_decoder_map', 'decoder_weight_scale': DECODER_WEIGHT_SCALE,
    'decoder_bias_init': 'copied_from_v5_vietnamese_unigram_logfreq' if BASELINE_FREQ_BIAS else 'zero',
    'anchor_pairs': 0,   # override v5's 4989 — this artifact carries NO anchor semantics
})
print('Train it: set nb02 BASE_MODEL_REF =', f"'init/{BASELINE_NAME}/model'")

  neobert_random_init_baseline: model.encoder.weight swapped, roundtrip max|Δ|=0.00e+00
  neobert_random_init_baseline: decoder.weight swapped, roundtrip max|Δ|=0.00e+00
  neobert_random_init_baseline: decoder.bias swapped, roundtrip max|Δ|=0.00e+00
Train it: set nb02 BASE_MODEL_REF = 'init/neobert_random_init_baseline/model'


## 5. Health gate — loads + forward FINITE on CUDA (rotary fix in effect)

In [ ]:
# ── Comparative health gate: v5 control vs random baseline, same session/batch ──
# Distinguishes environment NaN (v5 also fails -> wrong runtime/stack, artifact fine)
# from artifact NaN (v5 finite, baseline fails -> the random embeddings trigger it).
for cdir in glob.glob('/root/.cache/huggingface/modules/transformers_modules/*'):
    if Path(cdir).is_dir():
        shutil.rmtree(cdir, ignore_errors=True)
for k in [k for k in sys.modules if k.startswith('transformers_modules')]:
    del sys.modules[k]
importlib.invalidate_caches()

print('env:', torch.__version__, '| cuda', torch.version.cuda,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

SENTS = ['Việt Nam là một quốc gia ở Đông Nam Á.',
         'Hôm nay thời tiết rất đẹp và trời trong xanh.',
         'Kinh tế Việt Nam tăng trưởng trong năm qua.',
         'Trẻ em cần được tiêm phòng đầy đủ để tránh bệnh.']

def gate_forward(model_dir, tag):
    m = AutoModelForMaskedLM.from_pretrained(model_dir, trust_remote_code=True).to(DEVICE).eval()
    t = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    enc = t(SENTS, padding=True, truncation=True, max_length=48, return_tensors='pt').to(DEVICE)
    ids = enc['input_ids']; torch.manual_seed(0)
    pm = torch.full(ids.shape, 0.2)
    for sid in set(t.all_special_ids):
        pm[ids.cpu() == sid] = 0
    msk = torch.bernoulli(pm).bool().to(DEVICE)
    lab = torch.full_like(ids, -100); lab[msk] = ids[msk]
    mids = ids.clone(); mids[msk] = t.mask_token_id
    wfin = all(bool(torch.isfinite(prm).all()) for prm in m.parameters())

    first_nan = []  # name of the first module whose output goes non-finite
    def mk_hook(name):
        def hook(_mod, _inp, out):
            if first_nan:
                return
            outs = out if isinstance(out, (tuple, list)) else (out,)
            for o in outs:
                if isinstance(o, torch.Tensor) and o.is_floating_point() and not torch.isfinite(o).all():
                    first_nan.append(name)
                    return
        return hook
    handles = [mod.register_forward_hook(mk_hook(n)) for n, mod in m.named_modules() if n]
    with torch.no_grad():
        lg = m(input_ids=mids, attention_mask=enc['attention_mask']).logits
    for h in handles:
        h.remove()
    fin = bool(torch.isfinite(lg).all())
    loss = F.cross_entropy(lg.reshape(-1, lg.size(-1)).float(), lab.reshape(-1), ignore_index=-100).item()
    print(f'{tag:18s} weights_finite={str(wfin):5s} forward_finite={str(fin):5s} '
          f'loss={loss:7.3f}  first_nan={first_nan[0] if first_nan else "-"}')
    del m; torch.cuda.empty_cache()
    return fin

ok_v5 = gate_forward(SALT_MODEL, 'v5 control')
ok_bl = gate_forward(MODEL_DIR, 'random baseline')
if ok_v5 and ok_bl:
    print('✅ both finite — baseline ready for CPT (nb02 BASE_MODEL_REF = init/%s/model)' % BASELINE_NAME)
elif not ok_v5:
    print('❌ v5 ALSO NaN here → ENVIRONMENT problem (this runtime differs from the one nb12 certified); '
          'artifact is probably fine — rerun this cell on the training runtime.')
else:
    print('❌ baseline NaN while v5 finite in the SAME session → the random embeddings trigger it; '
          'report first_nan module above.')


## 6. L4 SDPA bisect — which attention path NaNs on this GPU?

The gate showed v5 AND baseline NaN at `transformer_encoder.0.wo` on L4 with patched
rotary/SwiGLU, while the identical artifact is finite on A100. The only op between `qkv` (finite)
and `wo` (first NaN) not yet exonerated is `F.scaled_dot_product_attention` itself — functional,
so module hooks can't see it; `wo` merely inherits its NaN. NeoBERT calls it with
`attn_mask=attention_mask.bool()`; a masked-SDPA kernel bug on this GPU/stack would explain
everything, including why nb11's "random q,k,v through SDPA" control (likely run mask-less)
wrongly cleared SDPA. Each line below isolates one variable.

In [ ]:
# run on the L4 runtime that just failed the gate
from torch.nn.attention import sdpa_kernel, SDPBackend

m = AutoModelForMaskedLM.from_pretrained(SALT_MODEL, trust_remote_code=True).to(DEVICE).eval()
t = AutoTokenizer.from_pretrained(SALT_MODEL, trust_remote_code=True)
enc  = t(SENTS, padding=True, truncation=True, max_length=48, return_tensors='pt').to(DEVICE)
enc1 = t(SENTS[:1], return_tensors='pt').to(DEVICE)   # no padding -> all-ones mask

@torch.no_grad()
def fwd_ok(e, **kw):
    try:
        lg = m(input_ids=e['input_ids'], attention_mask=e['attention_mask'], **kw).logits
        return str(bool(torch.isfinite(lg).all()))
    except Exception as ex:
        return f'ERR {type(ex).__name__}: {ex}'[:90]

print('padded batch, default backends :', fwd_ok(enc))
print('single seq, all-ones mask      :', fwd_ok(enc1))
for be in (SDPBackend.MATH, SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.CUDNN_ATTENTION):
    try:
        with sdpa_kernel([be]):
            r = fwd_ok(enc)
    except Exception as ex:
        r = f'ERR {type(ex).__name__}'
    print(f'padded batch, {be.name:19s}:', r)
print('eager path (output_attentions) :', fwd_ok(enc, output_attentions=True))
tf32 = torch.backends.cuda.matmul.allow_tf32
torch.backends.cuda.matmul.allow_tf32 = False
print('padded batch, TF32 OFF         :', fwd_ok(enc))
torch.backends.cuda.matmul.allow_tf32 = tf32
del m; torch.cuda.empty_cache()

# Reading the table:
#  - MATH True / EFFICIENT False  -> mem-efficient masked-SDPA kernel broken on this GPU:
#    fix = disable that backend before forward/training on this runtime class:
#      torch.backends.cuda.enable_mem_efficient_sdp(False)
#  - all-ones mask True / padded False -> padding-content issue, not kernel
#  - TF32 OFF True                  -> TF32 matmul kernel, keep TF32 off on L4
#  - everything False               -> deeper stack issue: train on A100 only